# M5 Report Visualizations

This notebook creates the stakeholder-facing figures used in the M5 Data Modelling & Visualisation report. It reads the saved model outputs from `M5_Data_Modelling_and_Visualisation_Report/model_outputs` and writes PNG files to `M5_Data_Modelling_and_Visualisation_Report/figures`.

In [ ]:
# Run once if the environment does not already contain these packages.
%pip install -q pandas pillow

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "M5_Data_Modelling_and_Visualisation_Report" / "model_outputs" / "model_comparison.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing M5_Data_Modelling_and_Visualisation_Report/model_outputs/model_comparison.csv")

ROOT = find_project_root()
OUTPUT_DIR = ROOT / "M5_Data_Modelling_and_Visualisation_Report" / "model_outputs"
FIGURE_DIR = ROOT / "M5_Data_Modelling_and_Visualisation_Report" / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

comparison = pd.read_csv(OUTPUT_DIR / "model_comparison.csv")
ft_report = pd.read_csv(OUTPUT_DIR / "ft_transformer_calibrated_test_classification_report.csv", index_col=0)
catboost_importance = pd.read_csv(OUTPUT_DIR / "catboost_feature_importance.csv")
ft_predictions = pd.read_csv(OUTPUT_DIR / "ft_transformer_calibrated_test_predictions.csv")

print("Loaded model comparison rows:", len(comparison))
print("Figure output directory:", FIGURE_DIR)

In [ ]:
def load_font(size, bold=False):
    candidates = [
        Path("C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf"),
        Path("C:/Windows/Fonts/calibrib.ttf" if bold else "C:/Windows/Fonts/calibri.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
    ]
    for path in candidates:
        if path.exists():
            return ImageFont.truetype(str(path), size=size)
    return ImageFont.load_default()

FONT_TITLE = load_font(30, bold=True)
FONT_SUBTITLE = load_font(20)
FONT_AXIS = load_font(18)
FONT_LABEL = load_font(16)
FONT_SMALL = load_font(14)

BLUE = "#4C78A8"
ORANGE = "#F58518"
GREEN = "#54A24B"
RED = "#E45756"
GRAY = "#555555"
LIGHT_GRID = "#DDDDDD"
TEXT = "#222222"
BACKGROUND = "#FFFFFF"

def text_size(draw, text, font):
    box = draw.textbbox((0, 0), str(text), font=font)
    return box[2] - box[0], box[3] - box[1]

def draw_wrapped_text(draw, xy, text, font, fill, max_width, line_gap=4):
    words = str(text).split()
    lines = []
    current = ""
    for word in words:
        trial = word if not current else current + " " + word
        if text_size(draw, trial, font)[0] <= max_width:
            current = trial
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    x, y = xy
    for line in lines:
        draw.text((x, y), line, font=font, fill=fill)
        y += text_size(draw, line, font)[1] + line_gap
    return y

def save_grouped_bar_chart(path, title, categories, series, y_max=1.0):
    width, height = 1200, 720
    margin_left, margin_right = 110, 60
    margin_top, margin_bottom = 115, 145
    plot_w = width - margin_left - margin_right
    plot_h = height - margin_top - margin_bottom
    image = Image.new("RGB", (width, height), BACKGROUND)
    draw = ImageDraw.Draw(image)
    draw.text((margin_left, 35), title, font=FONT_TITLE, fill=TEXT)

    for i in range(6):
        value = y_max * i / 5
        y = margin_top + plot_h - (value / y_max) * plot_h
        draw.line((margin_left, y, width - margin_right, y), fill=LIGHT_GRID, width=1)
        draw.text((35, y - 10), f"{value:.1f}", font=FONT_SMALL, fill=GRAY)

    draw.line((margin_left, margin_top, margin_left, margin_top + plot_h), fill=GRAY, width=2)
    draw.line((margin_left, margin_top + plot_h, width - margin_right, margin_top + plot_h), fill=GRAY, width=2)

    group_w = plot_w / len(categories)
    bar_gap = 8
    bar_w = min(70, (group_w - 50) / len(series))
    for c_idx, category in enumerate(categories):
        center = margin_left + group_w * c_idx + group_w / 2
        start_x = center - ((bar_w * len(series)) + (bar_gap * (len(series) - 1))) / 2
        for s_idx, item in enumerate(series):
            value = item["values"][c_idx]
            x0 = start_x + s_idx * (bar_w + bar_gap)
            x1 = x0 + bar_w
            y1 = margin_top + plot_h
            y0 = y1 - (value / y_max) * plot_h
            draw.rectangle((x0, y0, x1, y1), fill=item["color"])
            label = f"{value:.3f}"
            tw, th = text_size(draw, label, FONT_SMALL)
            draw.text((x0 + (bar_w - tw) / 2, y0 - th - 5), label, font=FONT_SMALL, fill=TEXT)
        label_w = min(int(group_w - 20), 250)
        y_after = draw_wrapped_text(draw, (center - label_w / 2, margin_top + plot_h + 22), category, FONT_LABEL, TEXT, label_w)

    legend_y = height - 48
    legend_x = margin_left
    for item in series:
        draw.rectangle((legend_x, legend_y, legend_x + 22, legend_y + 14), fill=item["color"])
        draw.text((legend_x + 32, legend_y - 4), item["name"], font=FONT_LABEL, fill=TEXT)
        legend_x += text_size(draw, item["name"], FONT_LABEL)[0] + 80

    image.save(path)
    return path

def save_horizontal_bar_chart(path, title, labels, values):
    width, height = 1200, 760
    margin_left, margin_right = 360, 70
    margin_top, margin_bottom = 110, 60
    plot_w = width - margin_left - margin_right
    plot_h = height - margin_top - margin_bottom
    image = Image.new("RGB", (width, height), BACKGROUND)
    draw = ImageDraw.Draw(image)
    draw.text((margin_left, 35), title, font=FONT_TITLE, fill=TEXT)
    max_value = max(values) * 1.08
    row_h = plot_h / len(labels)
    for idx, (label, value) in enumerate(zip(labels, values)):
        y = margin_top + idx * row_h + row_h * 0.2
        bar_h = row_h * 0.55
        draw_wrapped_text(draw, (30, y - 4), label, FONT_LABEL, TEXT, margin_left - 55)
        x1 = margin_left + (value / max_value) * plot_w
        draw.rectangle((margin_left, y, x1, y + bar_h), fill=BLUE)
        draw.text((x1 + 10, y + 2), f"{value:.2f}", font=FONT_LABEL, fill=TEXT)
    draw.line((margin_left, margin_top - 5, margin_left, margin_top + plot_h), fill=GRAY, width=2)
    image.save(path)
    return path

def save_confusion_heatmap(path, title, matrix, labels):
    width, height = 980, 760
    image = Image.new("RGB", (width, height), BACKGROUND)
    draw = ImageDraw.Draw(image)
    draw.text((90, 35), title, font=FONT_TITLE, fill=TEXT)
    left, top = 250, 140
    cell = 145
    for i, actual in enumerate(labels):
        total = sum(matrix[i])
        draw_wrapped_text(draw, (45, top + i * cell + 48), actual, FONT_LABEL, TEXT, 165)
        for j, predicted in enumerate(labels):
            count = matrix[i][j]
            pct = count / total if total else 0
            shade = int(245 - 165 * pct)
            color = (shade, shade + 5 if shade < 245 else 245, 255)
            x0, y0 = left + j * cell, top + i * cell
            draw.rectangle((x0, y0, x0 + cell, y0 + cell), fill=color, outline="#FFFFFF", width=3)
            value_text = f"{pct:.0%}\n{count:,}"
            line1, line2 = value_text.split("\n")
            for offset, line in enumerate([line1, line2]):
                tw, th = text_size(draw, line, FONT_AXIS if offset == 0 else FONT_LABEL)
                draw.text((x0 + (cell - tw) / 2, y0 + 48 + offset * 28), line, font=FONT_AXIS if offset == 0 else FONT_LABEL, fill=TEXT)
    for j, label in enumerate(labels):
        draw_wrapped_text(draw, (left + j * cell + 8, top - 70), label, FONT_LABEL, TEXT, cell - 16)
    draw.text((left + 70, height - 95), "Predicted injury class", font=FONT_SUBTITLE, fill=TEXT)
    draw.text((45, top - 35), "Actual class", font=FONT_SUBTITLE, fill=TEXT)
    image.save(path)
    return path

In [ ]:
test_models = comparison[(comparison["split"] == "test") & comparison["model"].str.endswith("raw")].copy()
test_models["model_family"] = test_models["model"].str.replace(" raw", "", regex=False)
test_models = test_models.sort_values("macro_f1", ascending=False)

figure1 = save_grouped_bar_chart(
    FIGURE_DIR / "figure1_model_performance_comparison.png",
    "Test Performance by Model",
    test_models["model_family"].tolist(),
    [
        {"name": "Accuracy", "values": test_models["accuracy"].tolist(), "color": BLUE},
        {"name": "Balanced accuracy", "values": test_models["balanced_accuracy"].tolist(), "color": ORANGE},
        {"name": "Macro F1", "values": test_models["macro_f1"].tolist(), "color": GREEN},
    ],
    y_max=0.85,
)
figure1

In [ ]:
class_rows = ft_report.loc[["NO_INJURY", "MINOR_INJURY", "SEVERE_INJURY"], ["precision", "recall", "f1-score"]]
figure2 = save_grouped_bar_chart(
    FIGURE_DIR / "figure2_best_model_class_performance.png",
    "FT-Transformer Performance by Injury Class",
    ["No injury", "Minor injury", "Severe injury"],
    [
        {"name": "Precision", "values": class_rows["precision"].tolist(), "color": BLUE},
        {"name": "Recall", "values": class_rows["recall"].tolist(), "color": ORANGE},
        {"name": "F1-score", "values": class_rows["f1-score"].tolist(), "color": GREEN},
    ],
    y_max=0.95,
)
figure2

In [ ]:
top_features = catboost_importance.head(10).iloc[::-1]
figure3 = save_horizontal_bar_chart(
    FIGURE_DIR / "figure3_catboost_feature_importance.png",
    "Top CatBoost Feature Importance",
    top_features["feature"].tolist(),
    top_features["importance"].tolist(),
)
figure3

In [ ]:
label_order = ["NO_INJURY", "MINOR_INJURY", "SEVERE_INJURY"]
display_labels = ["No injury", "Minor injury", "Severe injury"]
confusion = pd.crosstab(ft_predictions["actual_label"], ft_predictions["predicted_label"])
confusion = confusion.reindex(index=label_order, columns=label_order, fill_value=0)
figure4 = save_confusion_heatmap(
    FIGURE_DIR / "figure4_ft_transformer_confusion_matrix.png",
    "FT-Transformer Confusion Matrix",
    confusion.values.tolist(),
    display_labels,
)
figure4

In [ ]:
for path in sorted(FIGURE_DIR.glob("figure*.png")):
    print(path.relative_to(ROOT))